# 🎧 Customer Support Knowledge Assistant
## A Retrieval-Augmented Generation (RAG) Assistant with Conversational Memory

**Mid-term project — RAG + Conversational Memory, built on the Bitext Customer Support dataset**

---

## 1. Project Overview

This notebook builds an AI assistant that answers customer-support questions the way a trained support
agent would: by looking up the closest matching, previously-approved support answers and rephrasing them
into a natural reply — instead of inventing an answer from the language model's own general knowledge.

The knowledge source is the **Bitext Customer Support LLM Chatbot Training Dataset**
(`bitext/Bitext-customer-support-llm-chatbot-training-dataset`, Hugging Face Hub): 26,872 `(instruction,
category, intent, response)` records spanning **27 intents across 10 categories** (orders, refunds,
shipping, accounts, payments, invoices, feedback, subscriptions, contact and cancellation topics).

Every row already contains a **user question and the response a support agent should give**. That makes
this dataset unusually well suited to RAG: instead of chunking a PDF, we treat each `(instruction,
response)` pair as one atomic, pre-written knowledge unit and let the retriever find the pairs whose
*question phrasing* is closest to what the live user typed.

### What the system does

| Capability | Meaning in this project |
| --- | --- |
| **Data integration** | The Bitext dataset is loaded with 🤗 `datasets.load_dataset(...)` and becomes the only knowledge source. |
| **Knowledge-base construction** | Each `(instruction, category, intent, response)` row becomes one retrievable document. |
| **Embeddings** | Every document is converted into a vector with a free HuggingFace sentence-transformer model. |
| **Vector database** | Vectors are indexed in **FAISS** for fast semantic (meaning-based) search. |
| **Retrieval** | A LangChain retriever returns the support entries most relevant to the user's question. |
| **RAG** | The retrieved entries are inserted into a prompt and sent to an LLM, which writes the final answer. |
| **Conversational memory** | The full conversation is kept in a buffer so follow-up questions work. |
| **Follow-up handling** | The current question is rewritten into a standalone question using the conversation history *before* retrieval. |
| **Citations** | Every answer names the intent/category the answer was grounded in. |
| **Interface** | An in-notebook chat loop, plus a lightweight Gradio chat widget. |
| **Testing & evaluation** | Twelve test cases plus a qualitative evaluation table with an explicitly defined pass rule. |

### Why RAG instead of just asking an LLM

A general language model may invent a plausible-sounding but wrong support policy (a refund window that
doesn't exist, a cancellation process the company doesn't use). RAG constrains the model: it may only use
text retrieved from the approved support responses, and the prompt instructs it to say so when the
dataset has no matching entry. This is the same idea used in production support bots that must never
promise something the business hasn't actually approved.

### Knowledge source

`bitext/Bitext-customer-support-llm-chatbot-training-dataset` — 27 intents (e.g. `cancel_order`,
`track_order`, `get_refund`, `change_password`, `payment_issue`, `complaint`, `contact_customer_service`)
grouped into 10 categories (`ORDER`, `REFUND`, `SHIPPING`, `ACCOUNT`, `PAYMENT`, `INVOICE`, `FEEDBACK`,
`SUBSCRIPTION`, `CONTACT`, `CANCELLATION_FEE`).

The dataset deliberately contains **generic, templated answers** (placeholders such as
`{{Order Number}}`), not live account data. That makes it a good test of grounding: the assistant should
present the approved response pattern, not invent order numbers, refund amounts or dates that were never
in the data.


## 2. Architecture

```text
        bitext/Bitext-customer-support-llm-chatbot-training-dataset
                                    |
                                    v
                     datasets.load_dataset(...)["train"]
                                    |
                                    v
        DataFrame: flags | instruction | category | intent | response
                                    |
                                    v
      build_documents()  ->  one LangChain Document per (intent, instruction, response)
              (sampled per intent to keep the index small and fast)
                                    |
                                    v
                                 Documents
                                    |
                                    v
        HuggingFaceEmbeddings  (sentence-transformers/all-MiniLM-L6-v2)
                                    |
                                    v
                       FAISS  (in-memory vector index)
                                    |
                                    v
                     Retriever  (as_retriever, search_type=mmr, k=4)
                                    ^
                                    |
   User Question  +  Conversation History  -->  Contextualised standalone question
                                    |
                                    v
                    Retrieved Context (matching support entries)
                                    |
                                    v
        Prompt  (grounding rules + context + chat history + question)
                                    |
                                    v
                                   LLM
                                    |
                                    v
                    Final Answer  +  Category / Intent used
                                    |
                                    v
                    Conversation Memory  (buffer, full history)
```

### Component by component

1. **Bitext dataset** — the only knowledge source. Nothing outside it is treated as an approved policy.
2. **`build_documents()`** — turns each row into one `Document` whose text contains the category, intent,
   an example user phrasing, and the approved response. This is the *data-integration* and *chunking*
   step combined: because each row is already a short, self-contained knowledge unit, there is no PDF-style
   page-splitting to do — but we still sample and de-duplicate so the retrieval unit stays small and specific.
3. **Embeddings** — turn each document into a fixed-length vector that encodes meaning, so "I want my money
   back" can match a document written as "How can I obtain a refund for my purchase?".
4. **FAISS** — stores all document vectors and answers nearest-neighbour queries in milliseconds, locally
   and for free.
5. **Retriever** — the LangChain interface over FAISS; given a query string it returns the top `k` entries.
6. **Contextualisation** — rewrites a follow-up ("how long does that take?") into a standalone question
   ("how long does cancelling an order take?") using the chat history, *before* retrieval runs.
7. **LLM** — a *writer*, not a knowledge source: it rephrases the retrieved support entries into a natural
   reply and refuses when nothing relevant was retrieved.
8. **Conversation memory** — a full message buffer per session, so multi-turn conversations work.


## 3. Environment Setup

### 3.1 Install the required packages

**What this cell does:** installs the LangChain stack, FAISS, the Hugging Face `datasets` library, and the
libraries used for embeddings and the local language model.

**Why we need it:** Colab/Kaggle images ship with some of these but not all, and mixing LangChain versions
is the most common cause of `ImportError`. We pin the whole LangChain family to the compatible `0.3.x` line
so the modern APIs used later (`create_retrieval_chain`, `RunnableWithMessageHistory`) all exist.

**Expected output:** a short confirmation line with the installed LangChain version. Installation takes
1–3 minutes.


In [1]:
# ================================================================
# Package installation (run once per session)
# ================================================================
# NOTE: Colab/Kaggle images ship with a preinstalled `huggingface_hub` that is
# sometimes newer/older than what a plain "pip install --upgrade" resolves for
# transformers/datasets. A partial upgrade in place can leave mismatched files
# on disk (symptom: "cannot import name 'as_extended_path' from
# 'huggingface_hub.utils._paths'", or similar ImportErrors from
# transformers/datasets right after installing). The fix is a CLEAN reinstall
# of the huggingface stack (uninstall, then install pinned, compatible
# versions) rather than an in-place upgrade.
import subprocess
import sys

CORE_HF_STACK = [
    "huggingface_hub>=0.26.0,<0.28.0",
    "transformers>=4.46.0,<4.47.0",
    "tokenizers>=0.20.0,<0.21.0",
    "datasets>=3.0.0,<3.2.0",
    "accelerate>=1.0.0,<1.2.0",
]

REQUIRED_PACKAGES = [
    "langchain>=0.3.20,<0.4",
    "langchain-core>=0.3.40,<0.4",
    "langchain-community>=0.3.20,<0.4",
    "langchain-huggingface>=0.1.2,<0.4",
    "langchain-text-splitters>=0.3.5,<0.4",
    "faiss-cpu>=1.8.0",
    "sentence-transformers>=3.0.0",
    "gradio>=4.36.0",
]


def run_pip(args):
    command = [sys.executable, "-m", "pip", *args]
    process = subprocess.run(command, capture_output=True, text=True)
    if process.returncode != 0:
        print("[ERROR] pip command failed:", " ".join(args))
        print("\n".join(process.stderr.strip().splitlines()[-15:]))
        return False
    return True


print("Step 1/2: cleanly reinstalling the huggingface_hub / transformers / "
      "tokenizers / datasets / accelerate stack (this avoids version-mismatch "
      "ImportErrors from a partial in-place upgrade)...")
run_pip(["uninstall", "-y", "-q",
         "huggingface_hub", "transformers", "tokenizers", "datasets", "accelerate"])
CORE_OK = run_pip(["install", "--quiet", "--no-cache-dir", *CORE_HF_STACK])

print("Step 2/2: installing the LangChain / FAISS / Gradio stack...")
REST_OK = run_pip(["install", "--quiet", "--upgrade", *REQUIRED_PACKAGES])

INSTALL_OK = CORE_OK and REST_OK

if INSTALL_OK:
    import importlib
    import langchain

    importlib.reload(langchain)
    print("[OK] Packages installed.")
    print("LangChain version:", langchain.__version__)
    print("\n[IMPORTANT] If the very next cell still raises an ImportError")
    print("mentioning huggingface_hub / transformers, choose")
    print("'Runtime -> Restart session' (Colab) or 'Kernel -> Restart' (Kaggle)")
    print("ONCE, then re-run the notebook from the top WITHOUT re-running pip")
    print("install twice in a row. This is a one-time Colab/Kaggle quirk caused")
    print("by packages that were already imported by the platform before this")
    print("cell ran, not a problem with the pinned versions themselves.")
else:
    print("[ERROR] Installation failed. Re-run this cell (transient network "
          "errors are common), or check 'Internet' is ON in the notebook settings.")


Step 1/2: cleanly reinstalling the huggingface_hub / transformers / tokenizers / datasets / accelerate stack (this avoids version-mismatch ImportErrors from a partial in-place upgrade)...
Step 2/2: installing the LangChain / FAISS / Gradio stack...
[OK] Packages installed.
LangChain version: 0.3.30

[IMPORTANT] If the very next cell still raises an ImportError
mentioning huggingface_hub / transformers, choose
'Runtime -> Restart session' (Colab) or 'Kernel -> Restart' (Kaggle)
ONCE, then re-run the notebook from the top WITHOUT re-running pip
install twice in a row. This is a one-time Colab/Kaggle quirk caused
by packages that were already imported by the platform before this
cell ran, not a problem with the pinned versions themselves.


### 3.2 Imports and hardware detection

**What this cell does:** imports everything the notebook needs and detects whether a GPU is available.

**Why we need it:** GPU detection lets us automatically pick a stronger local language model when a GPU
is enabled, and a small CPU-friendly model otherwise — the notebook runs either way.


In [2]:
# ================================================================
# Imports
# ================================================================
import os
import re
import time
import textwrap
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    pipeline,
)

# --- LangChain: documents, splitting, embedding, storing ---
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline

# --- LangChain: prompting, chaining, memory ---
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.language_models.llms import LLM
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory

try:
    from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
except ImportError:  # older langchain-core naming
    from langchain_core.chat_history import BaseChatMessageHistory
    from langchain_community.chat_message_histories import (
        ChatMessageHistory as InMemoryChatMessageHistory,
    )

print("[OK] All imports successful")

# ---------------- Hardware detection ----------------
GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = "cuda" if GPU_AVAILABLE else "cpu"

print("PyTorch version :", torch.__version__)
print("Device          :", DEVICE)
if GPU_AVAILABLE:
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    print("GPU             : not enabled (the notebook still runs; a smaller LLM will be used)")


[OK] All imports successful
PyTorch version : 2.10.0+cu128
Device          : cuda
GPU             : Tesla T4


## 4. Configuration

**What this cell does:** defines every tunable setting of the project in **one place**.

**Why we need it:** no other cell contains a hard-coded dataset name, model name or hyper-parameter — they
all read these constants. That is what the project rubric calls "one place to edit".


In [3]:
# ==========================================================================
# PROJECT CONFIGURATION  --  the only cell you normally need to edit
# ==========================================================================

# ---- 1. Knowledge source --------------------------------------------------
DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
MAX_EXAMPLES_PER_INTENT = 40   # de-duplicated (instruction, response) pairs kept per intent

# ---- 2. Chunking / safety-net splitting ----------------------------------
CHUNK_SIZE = 400          # characters per chunk (rows are already short)
CHUNK_OVERLAP = 60        # characters repeated between neighbouring chunks

# ---- 3. Embeddings --------------------------------------------------------
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # free, fast, 384 dimensions

# ---- 4. Retrieval ----------------------------------------------------------
RETRIEVER_SEARCH_TYPE = "mmr"   # "mmr" (diverse results) or "similarity" (pure nearest neighbour)
RETRIEVER_K = 4                  # number of entries handed to the LLM
RETRIEVER_FETCH_K = 15           # candidates inspected before MMR selects k of them

# Minimum cosine similarity (embeddings are L2-normalised, so this is a true
# cosine, not a raw FAISS distance) the BEST retrieved entry must reach before
# the question is even sent to the LLM. Below this, the assistant refuses
# instead of asking a weak local model to "decide" whether the context is
# relevant -- this is what stops answers built on an irrelevant match (e.g. a
# "free upgrade" question weakly matching an unrelated intent).
#
# 0.35 matches the interpretation guide printed in the retrieval-metrics
# section of this notebook ("< 0.35 = weak match, typically out of scope").
# The earlier value of 0.20 was inconsistent with that guide and let clearly
# out-of-scope requests ("90% discount", "write a C++ program") slip past the
# gate on a weak lexical match, instead of hitting the fixed refusal path.
MIN_SIMILARITY = 0.35

# ---- 5. Language model ----------------------------------------------------
LOCAL_LLM_GPU = "Qwen/Qwen2.5-1.5B-Instruct"   # used when a GPU accelerator is enabled
LOCAL_LLM_CPU = "google/flan-t5-base"          # used on CPU-only sessions (small + reliable)
MAX_NEW_TOKENS = 300
TEMPERATURE = 0.1               # kept for reference only - generation now always uses
                                 # greedy decoding (do_sample=False), see section 13, for reproducibility

# ---- 6. Optional hosted API model (never hard-code a key!) ---------------
# Looked up from (in order): Colab userdata -> Kaggle Secrets -> environment variable.
API_KEY_SECRET_NAME = "GROQ_API_KEY"
GROQ_MODEL = "llama-3.3-70b-versatile"

# ---- 7. Conversation -------------------------------------------------------
SESSION_ID = "midterm-demo"

# ---- 8. Output --------------------------------------------------------------
WORKING_DIR = Path("/content") if Path("/content").exists() else Path(".")

print("=" * 70)
print("PROJECT CONFIGURATION")
print("=" * 70)
print(f"Dataset             : {DATASET_NAME}")
print(f"Examples per intent : {MAX_EXAMPLES_PER_INTENT}")
print(f"Embedding model     : {EMBEDDING_MODEL_NAME}")
print(f"Retriever           : search_type={RETRIEVER_SEARCH_TYPE}, k={RETRIEVER_K}")
print(f"Relevance threshold : {MIN_SIMILARITY} (min cosine before the LLM is even called)")
print(f"Local LLM (GPU/CPU) : {LOCAL_LLM_GPU} / {LOCAL_LLM_CPU}")
print(f"Device detected     : {DEVICE}")
print("=" * 70)


PROJECT CONFIGURATION
Dataset             : bitext/Bitext-customer-support-llm-chatbot-training-dataset
Examples per intent : 40
Embedding model     : sentence-transformers/all-MiniLM-L6-v2
Retriever           : search_type=mmr, k=4
Relevance threshold : 0.35 (min cosine before the LLM is even called)
Local LLM (GPU/CPU) : Qwen/Qwen2.5-1.5B-Instruct / google/flan-t5-base
Device detected     : cuda


### 4.1 Optional: using an API-based LLM

**The notebook works with no API key at all** — that is the default and recommended path. A hosted model
gives noticeably more fluent answers, but is entirely optional.

**No key is ever written in the notebook.** It is read from Colab's `userdata` secrets, Kaggle Secrets, or
an environment variable — whichever is available. The cell below only *checks* for a secret; it never
prints the key itself.


In [4]:
# ================================================================
# Read an optional API key from whichever secret store is available
# ================================================================
def get_secret(secret_name):
    # Order of preference: Colab userdata -> Kaggle Secrets -> environment variable -> None.
    try:
        from google.colab import userdata

        value = userdata.get(secret_name)
        if value:
            return value.strip()
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(secret_name)
        if value:
            return value.strip()
    except Exception:
        pass
    return os.environ.get(secret_name)


API_KEY = get_secret(API_KEY_SECRET_NAME)

if API_KEY:
    print(f"[OK] Secret '{API_KEY_SECRET_NAME}' found (length {len(API_KEY)}).")
else:
    print(f"[INFO] No secret named '{API_KEY_SECRET_NAME}' found.")
    print("       This is fine: the notebook will run a free local HuggingFace model instead.")


[INFO] No secret named 'GROQ_API_KEY' found.
       This is fine: the notebook will run a free local HuggingFace model instead.


## 5. Loading the Bitext Customer Support Dataset

**What this cell does:** downloads the dataset from the Hugging Face Hub and converts the `train` split
into a `pandas.DataFrame`.

**Why we need it:** this is the *data-integration* step of the project. Everything downstream — documents,
embeddings, retrieval, citations — depends on this.

**Expected output:** the number of rows loaded and the first few records.


In [5]:
# ================================================================
# Dataset loading
# ================================================================
ds = load_dataset(DATASET_NAME)

df = ds["train"].to_pandas()

print(f"[OK] Loaded '{DATASET_NAME}'")
print(f"Rows            : {len(df):,}")
print(f"Columns         : {list(df.columns)}")
df.head(3)


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

[OK] Loaded 'bitext/Bitext-customer-support-llm-chatbot-training-dataset'
Rows            : 26,872
Columns         : ['flags', 'instruction', 'category', 'intent', 'response']


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...


## 6. Dataset Inspection

**What this cell does:** prints basic statistics — categories, intents, and how many rows each has — plus a
few sample rows.

**Why we need it:** before building anything on top of this data we must understand its shape: how many
distinct intents exist, how many categories they group into, and what a typical `(instruction, response)`
pair looks like.

**Expected output:** a per-category intent count, and three sample rows.


In [6]:
# ================================================================
# Dataset inspection
# ================================================================
print("=" * 70)
print("DATASET STATISTICS")
print("=" * 70)
print(f"Total rows       : {len(df):,}")
print(f"Categories ({df['category'].nunique()}) : {sorted(df['category'].unique())}")
print(f"Intents    ({df['intent'].nunique()})")

intent_counts = (
    df.groupby(["category", "intent"]).size().reset_index(name="rows").sort_values(["category", "intent"])
)
display(intent_counts)

print("\nSample rows:")
for _, row in df.sample(3, random_state=42).iterrows():
    print("-" * 100)
    print(f"Category   : {row['category']}")
    print(f"Intent     : {row['intent']}")
    print(f"Instruction: {row['instruction']}")
    print(f"Response   : {row['response'][:300]}")


DATASET STATISTICS
Total rows       : 26,872
Categories (11) : ['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']
Intents    (27)


,category,intent,rows
0,ACCOUNT,create_account,997
1,ACCOUNT,delete_account,995
2,ACCOUNT,edit_account,1000
3,ACCOUNT,recover_password,995
4,ACCOUNT,registration_problems,999
5,ACCOUNT,switch_account,1000
6,CANCEL,check_cancellation_fee,950
7,CONTACT,contact_customer_service,1000
8,CONTACT,contact_human_agent,999
9,DELIVERY,delivery_options,995



Sample rows:
----------------------------------------------------------------------------------------------------
Category   : CONTACT
Intent     : contact_human_agent
Instruction: I can't talk with  a human agent
Response   : Thank you for reaching out! I'm sorry to hear that you're facing difficulties in connecting with a human agent. Please rest assured that I'm here to assist you and resolve this issue. Could you please provide me with some more details about the problem you're experiencing? This will help me find the
----------------------------------------------------------------------------------------------------
Category   : INVOICE
Intent     : check_invoice
Instruction: I have got to locate hte bills from {{Person Name}}
Response   : I'll get right on it! I understand your need to locate the bills associated with {{Person Name}}. To assist you in finding the relevant invoices, could you please provide me with some additional details such as the date range or any other speci

## 7. Building the Knowledge-Base Documents

### Why this replaces PDF chunking

A PDF-based assistant must split long pages into overlapping chunks because a page mixes many topics and
overflows the embedding model's context window. This dataset does not have that problem: **every row is
already a single, self-contained knowledge unit** — one user phrasing and one approved response, about
exactly one intent. So the "chunk" here *is* the row; there is no page to split.

What we do need is **sampling**. With 26,872 rows and ~1,000 near-duplicate phrasings per intent, embedding
every single row would take a long time and would not improve retrieval quality (many rows are template
variations of the same sentence). We keep a de-duplicated sample of up to `MAX_EXAMPLES_PER_INTENT`
distinct `(instruction, response)` pairs per intent — enough to cover the different ways a user might ask,
small enough to embed and index in seconds.

### Document format

Each `Document.page_content` bundles the category, intent, an example user phrasing and the approved
response into one short paragraph. This means the retriever matches on the *user's phrasing* (via the
embedded instruction text) while the LLM receives the *approved response* in the same retrieved chunk — no
separate lookup step is needed.

**Expected output:** the number of documents built and their distribution across categories.


In [7]:
# ================================================================
# Knowledge-base document construction
# ================================================================
NUMBERED_STEP_RE = re.compile(r"^\s*(\d+)\.\s")


def is_well_formed_response(response):
    # Reject a response whose text is a numbered-list FRAGMENT - e.g. a step
    # "3. Identify the Specific Purchase..." with no "1." or "2." before it.
    # This is a known data-quality artifact in the raw Bitext dataset (a small
    # number of rows contain only the tail of a longer list). Indexing one of
    # these as a knowledge-base document produces an answer that jumps
    # straight to step 3 with no 1-2, which is confusing to a real user.
    match = NUMBERED_STEP_RE.match(str(response).strip())
    return not (match and match.group(1) != "1")


def build_documents(dataframe, max_per_intent=MAX_EXAMPLES_PER_INTENT):
    # Turn a de-duplicated, capped sample of (instruction, response) rows per intent
    # into one retrievable LangChain Document each.
    documents = []
    deduplicated = dataframe.drop_duplicates(subset=["instruction", "response"])
    deduplicated = deduplicated[deduplicated["response"].apply(is_well_formed_response)]
    grouped = deduplicated.groupby("intent")

    for intent, group in grouped:
        sampled = group.sample(min(len(group), max_per_intent), random_state=42)
        for position, (_, row) in enumerate(sampled.iterrows()):
            content = (
                f"Category: {row['category']}\n"
                f"Intent: {row['intent']}\n"
                f"Example user question: {row['instruction']}\n"
                f"Approved support response: {row['response']}"
            )
            documents.append(
                Document(
                    page_content=content,
                    metadata={
                        "category": row["category"],
                        "intent": row["intent"],
                        "instruction": row["instruction"],
                        "doc_id": f"{intent}-{position}",
                    },
                )
            )
    return documents


documents = build_documents(df)

print("=" * 70)
print("KNOWLEDGE BASE")
print("=" * 70)
print(f"Rows in source dataset      : {len(df):,}")
print(f"Documents built             : {len(documents):,}")
print(f"Intents covered             : {df['intent'].nunique()}")
print(f"Avg documents per intent    : {len(documents) / df['intent'].nunique():.1f}")
print("\nExample document:\n")
print(documents[0].page_content)
print("\nMetadata:", documents[0].metadata)

removed = (~df.drop_duplicates(subset=["instruction", "response"])["response"]
           .apply(is_well_formed_response)).sum()
print(f"\nRows dropped as malformed numbered-list fragments: {removed}")


KNOWLEDGE BASE
Rows in source dataset      : 26,872
Documents built             : 1,080
Intents covered             : 27
Avg documents per intent    : 40.0

Example document:

Category: ORDER
Intent: cancel_order
Example user question: I cannot afford purchase {{Order Number}}
Approved support response: I understand the financial constraints you are facing and the need to cancel purchase {{Order Number}}. Our goal is to assist you in the best way possible. To proceed with the cancellation, please follow these steps:

1. Sign in to Your Account: Access our {{Online Company Portal Info}} by signing in with your credentials.

2. Navigate to Your Orders: Once you are signed in, locate the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.

3. Find the Relevant Purchase: Look for the purchase associated with the order number {{Order Number}} and click on it for more details.

4. Initiate Cancellation: You will see an option labeled '{{Online Order Interaction}}.' Plea

## 8. Safety-Net Chunking

**What this cell does:** runs the documents through `RecursiveCharacterTextSplitter` anyway, with a chunk
size comfortably larger than a typical document.

**Why we still do this:** almost every document already fits in one chunk (support responses are short),
but a handful of intents have longer, multi-sentence responses. Running the splitter guarantees no chunk
silently exceeds the embedding model's effective input length, without changing the vast majority of
documents at all.

**Expected output:** confirmation that the number of chunks is very close to the number of source documents.


In [8]:
# ================================================================
# Safety-net splitting
# ================================================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
)

chunks = text_splitter.split_documents(documents)

for position, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = position

chunk_lengths = [len(chunk.page_content) for chunk in chunks]
print("=" * 70)
print("CHUNKING RESULT")
print("=" * 70)
print(f"Documents in   : {len(documents):,}")
print(f"Chunks out     : {len(chunks):,}")
print(f"Average length : {sum(chunk_lengths) / len(chunk_lengths):,.0f} characters")
print(f"Longest chunk  : {max(chunk_lengths)} characters")


CHUNKING RESULT
Documents in   : 1,080
Chunks out     : 3,527
Average length : 242 characters
Longest chunk  : 400 characters


## 9. Creating Embeddings

### Why semantic similarity matters here

A customer will not type the dataset's exact vocabulary. They type:

| User writes | Knowledge base says |
| --- | --- |
| "I want my money back" | "How can I obtain a refund for my purchase?" |
| "cancel my order please" | "I need to cancel an order" |
| "where's my package" | "how do I track my order" |

Plain keyword search fails on all three rows. Embeddings succeed, because "money back" and "refund" are
near each other in meaning space.

### Why `sentence-transformers/all-MiniLM-L6-v2`

* **Free and local** — no API key, no cost, no rate limit.
* **Small and fast** — ~90 MB, embeds the whole knowledge base in seconds on CPU.
* **Well suited to the task** — trained specifically for sentence/passage similarity search.

We set `normalize_embeddings=True`, which makes every vector unit length, turning FAISS's L2 distance into
a monotone function of cosine similarity.

**Expected output:** the embedding dimension, timing information, and a demonstration that semantically
related sentences score higher than unrelated ones.


In [9]:
# ================================================================
# Embedding model
# ================================================================
def build_embeddings(model_name=EMBEDDING_MODEL_NAME, device=DEVICE):
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True},
    )


print("Loading embedding model (first run downloads ~90 MB)...")
start_time = time.time()
embeddings = build_embeddings()
print(f"[OK] Embedding model ready in {time.time() - start_time:.1f} s")

sample_vector = embeddings.embed_query("I want my money back for a broken item")
print(f"Embedding dimension : {len(sample_vector)}")


Loading embedding model (first run downloads ~90 MB)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[OK] Embedding model ready in 11.1 s
Embedding dimension : 384


### 9.1 Demonstrating semantic similarity

**What this cell does:** embeds one customer message and several candidate support-entry phrasings, then
ranks the candidates by cosine similarity.

**Why we need it:** it proves — before FAISS is even involved — that the embedding model captures meaning
rather than shared keywords.


In [10]:
# ================================================================
# Semantic similarity demonstration
# ================================================================
def cosine_similarity(vector_a, vector_b):
    return sum(a * b for a, b in zip(vector_a, vector_b))


question = "I want my money back for something I bought"
candidate_sentences = [
    "How can I obtain a refund for my purchase?",
    "I need to cancel an order",
    "Is it possible to track the status of my delivery?",
    "How can I update my billing address?",
    "I would like to leave feedback about your service",
]

question_vector = embeddings.embed_query(question)
candidate_vectors = embeddings.embed_documents(candidate_sentences)

scored = sorted(
    ((cosine_similarity(question_vector, vector), sentence)
     for vector, sentence in zip(candidate_vectors, candidate_sentences)),
    reverse=True,
)

print(f"QUESTION: {question}\n")
print("Ranked by semantic similarity (1.0 = identical meaning):")
print("-" * 100)
for score, sentence in scored:
    print(f"{score:0.3f} | {sentence}")


QUESTION: I want my money back for something I bought

Ranked by semantic similarity (1.0 = identical meaning):
----------------------------------------------------------------------------------------------------
0.709 | How can I obtain a refund for my purchase?
0.450 | I need to cancel an order
0.213 | How can I update my billing address?
0.145 | I would like to leave feedback about your service
0.127 | Is it possible to track the status of my delivery?


## 10. Building the FAISS Vector Store

**What this cell does:** embeds every chunk and builds the FAISS index.

**Why FAISS:** it holds one vector per chunk together with its text and metadata (category, intent), and
answers nearest-neighbour queries in milliseconds, entirely in this notebook's memory — no server, no
account, no cost.

**Expected output:** the number of vectors indexed and the build time.


In [11]:
# ================================================================
# FAISS vector store
# ================================================================
def build_vectorstore(chunks, embeddings):
    if not chunks:
        raise ValueError("No chunks supplied. Re-run the document-building cell first.")
    return FAISS.from_documents(documents=chunks, embedding=embeddings)


print(f"Embedding {len(chunks)} chunks and building the FAISS index...")
start_time = time.time()
vectorstore = build_vectorstore(chunks, embeddings)
build_seconds = time.time() - start_time

print("=" * 70)
print("FAISS VECTOR STORE")
print("=" * 70)
print(f"Vectors indexed  : {vectorstore.index.ntotal}")
print(f"Vector dimension : {vectorstore.index.d}")
print(f"Build time       : {build_seconds:.1f} s")


Embedding 3527 chunks and building the FAISS index...
FAISS VECTOR STORE
Vectors indexed  : 3527
Vector dimension : 384
Build time       : 2.6 s


## 11. Testing Semantic Retrieval

**What this cell does:** runs raw similarity searches against FAISS and prints the retrieved entries *with
their scores* — before any LLM is involved.

**Why we need it:** if retrieval is wrong here, no prompt engineering downstream can fix it.

**Reading the scores:** FAISS returns an **L2 distance** (smaller = more similar). Because the vectors are
normalised, `cosine_similarity = 1 - (L2_distance ** 2) / 2`.


In [12]:
# ================================================================
# Raw similarity search against FAISS
# ================================================================
def l2_to_cosine(distance):
    return 1.0 - (distance ** 2) / 2.0


def show_similarity_search(query, k=RETRIEVER_K, snippet_chars=280):
    results = vectorstore.similarity_search_with_score(query, k=k)

    print("=" * 100)
    print(f"QUERY: {query}")
    print("=" * 100)

    for rank, (document, distance) in enumerate(results, start=1):
        text = re.sub(r"\s+", " ", document.page_content.strip())
        print(f"\n[{rank}] intent={document.metadata['intent']}  |  "
              f"category={document.metadata['category']}  |  cosine similarity {l2_to_cosine(distance):.3f}")
        print("-" * 100)
        print(textwrap.fill(text[:snippet_chars] + "...", width=100))

    return results


_ = show_similarity_search("I want a refund for a damaged product")


QUERY: I want a refund for a damaged product

[1] intent=check_refund_policy  |  category=REFUND  |  cosine similarity 0.761
----------------------------------------------------------------------------------------------------
Please keep in mind that individual vendors or service providers may have their own refund policies
and procedures. To get more specific information or assistance regarding your order or concern,
kindly provide me with the relevant details like the order number, product name, or ...

[2] intent=check_refund_policy  |  category=REFUND  |  cosine similarity 0.719
----------------------------------------------------------------------------------------------------
Please note that specific vendors or services may have their own refund policies, so we recommend
reviewing the terms and conditions of each product or service you purchase....

[3] intent=check_refund_policy  |  category=REFUND  |  cosine similarity 0.719
----------------------------------------------------

In [13]:
# Two more retrieval checks on different parts of the knowledge base.
_ = show_similarity_search("How do I cancel my order?", k=3)
print("\n\n")
_ = show_similarity_search("I forgot my account password", k=3)


QUERY: How do I cancel my order?

[1] intent=cancel_order  |  category=ORDER  |  cosine similarity 0.977
----------------------------------------------------------------------------------------------------
To cancel your order, please follow these steps:...

[2] intent=cancel_order  |  category=ORDER  |  cosine similarity 0.916
----------------------------------------------------------------------------------------------------
3. Locate the Purchase: In the order history, find the purchase with the order number {{Order
Number}} and click on it to view the details. 4. Initiate the Cancellation: Look for the '{{Online
Order Interaction}}' option associated with your purchase and select it. 5. Confirm the...

[3] intent=cancel_order  |  category=ORDER  |  cosine similarity 0.914
----------------------------------------------------------------------------------------------------
3. Identify the Specific Purchase: Look for the purchase associated with the order number {{Order
Number}} and c

## 12. The Retriever

**Why `search_type="mmr"` and `k=4`:**

* **MMR (Maximal Marginal Relevance)** first fetches `fetch_k` candidates, then picks `k` that are relevant
  *and* different from each other. This matters here because several intents overlap semantically
  (`cancel_order` and `change_order` are close in meaning); MMR avoids returning four near-duplicate
  entries for the same intent.
* **k = 4** is enough context to answer, few enough that a small LLM is not overwhelmed.

**Expected output:** the retrieved intents/categories for a test query.


In [14]:
# ================================================================
# LangChain retriever
# ================================================================
def make_retriever(vectorstore, search_type=RETRIEVER_SEARCH_TYPE, k=RETRIEVER_K, fetch_k=RETRIEVER_FETCH_K):
    search_kwargs = {"k": k}
    if search_type == "mmr":
        search_kwargs["fetch_k"] = fetch_k
        search_kwargs["lambda_mult"] = 0.6
    return vectorstore.as_retriever(search_type=search_type, search_kwargs=search_kwargs)


retriever = make_retriever(vectorstore)


def retrieve_documents(query, show=True, snippet_chars=220):
    retrieved = retriever.invoke(query)
    if show:
        print("=" * 100)
        print(f"USER QUESTION : {query}")
        print(f"ENTRIES RETURNED: {len(retrieved)}")
        print("=" * 100)
        for rank, document in enumerate(retrieved, start=1):
            text = re.sub(r"\s+", " ", document.page_content.strip())
            print(f"\n[{rank}] intent={document.metadata['intent']} (category={document.metadata['category']})")
            print("-" * 100)
            print(textwrap.fill(text[:snippet_chars] + "...", width=100))
    return retrieved


_ = retrieve_documents("Can you help me get my money back?")


USER QUESTION : Can you help me get my money back?
ENTRIES RETURNED: 4

[1] intent=get_refund (category=REFUND)
----------------------------------------------------------------------------------------------------
Approved support response: I can see that you need assistance with requesting a refund for your
money. I'm here to help you through the process and ensure that you have a seamless experience...

[2] intent=check_refund_policy (category=REFUND)
----------------------------------------------------------------------------------------------------
Our money back guarantee covers a wide range of situations, including: 1. Product Defect: If the
product you receive is defective or doesn't meet your expectations, you can request a refund. 2.
Service Dissatisfaction: I...

[3] intent=check_refund_policy (category=REFUND)
----------------------------------------------------------------------------------------------------
If you have any specific questions or concerns regarding our money 

## 13. Setting Up the LLM

**Why the model is chosen this way:**

| Situation | Model used | Why |
| --- | --- | --- |
| An API key is found and `API_KEY` is set | (left as an exercise / optional; see section 4.1) | best answer quality |
| GPU accelerator enabled (T4/P100/etc.) | `Qwen/Qwen2.5-1.5B-Instruct` | a genuine instruction-following chat model, small enough to load in seconds |
| CPU only | `google/flan-t5-base` | ~250M parameters, runs comfortably on CPU, follows extraction-style instructions well |
| Everything above fails | extractive fallback | clearly labelled; returns the most relevant retrieved sentence verbatim so the pipeline never crashes mid-demo |

**Important:** in a RAG system the LLM is a **writer, not a knowledge source**. Its job is to rephrase the
retrieved support entry into a friendly reply and to refuse when nothing relevant was retrieved. That is
why a small model is acceptable here, and why the grounding rules in the prompt matter more than model
size.

**Why the model is wrapped in `ChatHuggingFace`.** `Qwen2.5-Instruct` is a *chat*-tuned model: it was
fine-tuned on conversations formatted with its own chat template (special role tokens around `system`
/ `user` / `assistant` turns). If a `ChatPromptTemplate` is piped straight into a plain
`HuggingFacePipeline`, LangChain falls back to a generic `"System: ...\nHuman: ...\nAI:"` text dump that
does **not** match the tokens Qwen was trained on — the model then treats the prompt as an unfinished
transcript and happily keeps "completing" it with invented `User:` / `Assistant:` turns, emoji and all.
Wrapping the pipeline in `ChatHuggingFace(llm=..., tokenizer=...)` makes LangChain call
`tokenizer.apply_chat_template()` first, so Qwen receives the exact format it was trained on and stops
once its own turn is done. `flan-t5` has no chat template (it is instruction-tuned on plain text, not
turn-based chat) so it is left as a plain `HuggingFacePipeline` — that format already suits it.


In [15]:
# ================================================================
# Language model
# ================================================================
class ExtractiveFallbackLLM(LLM):
    # LAST-RESORT fallback used only if no real LLM can be loaded.
    @property
    def _llm_type(self):
        return "extractive_fallback"

    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        context = prompt.split("Context:")[-1].split("Human:")[0] if "Context:" in prompt else prompt
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", context) if len(s.strip()) > 20]
        if not sentences:
            return "I could not find that information in the customer support knowledge base."
        return sentences[0]


def build_local_llm():
    model_name = LOCAL_LLM_GPU if GPU_AVAILABLE else LOCAL_LLM_CPU
    print(f"Loading local model '{model_name}' on {DEVICE} (first run downloads the weights)...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    config = AutoConfig.from_pretrained(model_name)

    if getattr(config, "is_encoder_decoder", False):
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        model.to(DEVICE)
        text_pipeline = pipeline(
            "text2text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            truncation=True,
            device=0 if GPU_AVAILABLE else -1,
        )
        description = f"{model_name} (seq2seq, {DEVICE}, plain-text prompting)"
        # flan-t5 has no chat template - it is trained on plain instructions,
        # so a plain HuggingFacePipeline is the correct interface for it.
        return HuggingFacePipeline(pipeline=text_pipeline), description, False

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if GPU_AVAILABLE else torch.float32,
    )
    model.to(DEVICE)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    text_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=MAX_NEW_TOKENS,
        # Greedy decoding (do_sample=False), not sampling. A grounded support
        # answer should be reproducible: the same question, with the same
        # retrieved context, must always produce the same answer. TEMPERATURE
        # is kept in the configuration for reference but is no longer wired
        # into do_sample, since sampling was the reason identical questions
        # (e.g. "who are you") could get two different answers across calls.
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    base_llm = HuggingFacePipeline(pipeline=text_pipeline)

    if getattr(tokenizer, "chat_template", None):
        # Apply the model's own chat template so it stops once its turn ends,
        # instead of hallucinating extra User:/Assistant: turns.
        chat_model = ChatHuggingFace(llm=base_llm, tokenizer=tokenizer)
        description = f"{model_name} (causal, {DEVICE}, via ChatHuggingFace chat template)"
        return chat_model, description, True

    description = f"{model_name} (causal, {DEVICE}, no chat template available)"
    return base_llm, description, False


try:
    llm, LLM_DESCRIPTION, LLM_IS_CHAT = build_local_llm()
    print(f"[OK] LLM ready: {LLM_DESCRIPTION}")
except Exception as error:
    print(f"[WARN] Could not load a local LLM ({error}). Falling back to extractive answers.")
    llm = ExtractiveFallbackLLM()
    LLM_DESCRIPTION = "ExtractiveFallbackLLM (no generative model available)"
    LLM_IS_CHAT = False


Loading local model 'Qwen/Qwen2.5-1.5B-Instruct' on cuda (first run downloads the weights)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[OK] LLM ready: Qwen/Qwen2.5-1.5B-Instruct (causal, cuda, via ChatHuggingFace chat template)


In [16]:
# ---- Smoke test: confirm the model responds at all (not a RAG call yet) ----
try:
    probe = llm.invoke("Reply with exactly one short sentence: why is customer support important?")
    probe_text = probe.content if hasattr(probe, "content") else str(probe)
    print("Model responded:\n", textwrap.fill(probe_text.strip()[:400], width=100))
except Exception as error:
    print("[ERROR] The model did not respond:", error)


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Model responded:
 Customer support is crucial as it helps resolve issues promptly, enhances satisfaction, and fosters
long-term loyalty for businesses.


## 14. Building the RAG Pipeline

### The prompts

**1. The grounding / answering prompt.** This is where hallucination is controlled:

* answer **only** from the retrieved support entries;
* if nothing relevant was retrieved, say so with a fixed sentence instead of guessing;
* never invent a concrete order number, refund amount, date or account detail that was not in the
  retrieved text — the dataset only contains **template** responses (placeholders such as
  `{{Order Number}}`);
* keep the tone professional, empathetic and concise, the way a support agent would.

**2. The contextualisation prompt.** Turns a follow-up question into a standalone one before retrieval —
covered in section 15.


In [17]:
# ================================================================
# Prompts
# ================================================================
REFUSAL_SENTENCE = "I could not find an approved answer for that in the support knowledge base."

SYSTEM_PROMPT = '''You are a customer support assistant for an e-commerce company.

You must answer ONLY using the context below, which was retrieved from a knowledge base of
approved support responses (Bitext customer-support dataset).

RULES:
1. Use only the approved response(s) shown in the context. Never invent policies, refund
   windows, prices or dates that are not present in the context.
2. If the context section below is EMPTY, or does not actually answer the question, you MUST
   reply with EXACTLY this sentence and nothing else:
   \"''' + REFUSAL_SENTENCE + '''\"
   Do not soften it, do not add an apology before or after it, do not offer alternatives.
3. Never state a concrete order number, amount or date unless it literally appears in the
   context. Placeholders like {{Order Number}} may be kept as-is or referred to generically
   ("your order number").
4. Be concise, empathetic and professional. Two to five sentences is usually enough.
5. When useful, mention which topic the answer relates to (for example "About cancelling an
   order:").
6. Answer only as yourself, in a single turn. Never write "User:", "Human:", "Assistant:" or
   "AI:", and never continue the conversation with a new question of your own.

Context retrieved from the knowledge base:
{context}'''

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

CONTEXTUALIZE_PROMPT = '''Given the conversation history and the user's latest message,
rewrite the latest message as a single standalone question that can be understood without the
history. Replace pronouns such as "it", "that", "them" with the actual topic from the history.

Do NOT answer the question. Do NOT continue the conversation. Return ONLY the rewritten
question, as one line of text ending in a question mark, and nothing else - no labels, no
extra turns.
If the latest message is already standalone, return it unchanged.'''

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", CONTEXTUALIZE_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

print("[OK] Prompts created:")
print("  - qa_prompt            : grounding rules + retrieved context + chat history + question")
print("  - contextualize_prompt : rewrites follow-up questions into standalone questions")


[OK] Prompts created:
  - qa_prompt            : grounding rules + retrieved context + chat history + question
  - contextualize_prompt : rewrites follow-up questions into standalone questions


## 15. Follow-Up Question Handling (memory *inside* retrieval)

### The problem a naive RAG system has

```text
User: How can I cancel my order?
User: How long will that take?
```

The second question contains no mention of "order" or "cancel" at all. Sent straight to the retriever, it
matches almost nothing useful. **This is the single most common failure of naive RAG.**

### The solution used here

Memory is placed *before* retrieval:

```text
Conversation History + Current Question -> LLM rewrites -> Standalone question -> Retriever -> Answer
```

So "How long will that take?" becomes "How long does cancelling an order take?", which retrieves the
right entry.

### Two safety nets

* **Rewrite sanity guard.** Small local models sometimes answer the question instead of rewriting it, or
  invent a fake `AI:`/`Assistant:`/`User:` turn instead of a question. `sanitize_rewritten_query` rejects a
  rewrite that is too short, too long, does not end in a question mark, or contains one of those turn
  markers. If the LLM's rewrite is rejected, a **heuristic fallback** is used instead of silently reverting
  to the bare pronoun question: the current question is combined with the previous user turn
  ("`<question> (previous question was: <previous question>)`"), so the retriever still has a topic to
  match against even when the LLM rewrite fails.
* **Relevance gate.** Even a perfect rewrite does not guarantee a *relevant* match exists in the knowledge
  base. Before the LLM is asked to answer, the retriever's own best cosine similarity for the (rewritten)
  question is compared against `MIN_SIMILARITY`. Below that threshold the pipeline refuses immediately -
  no document is even handed to the LLM - which is what stops a weak question ("give me a free upgrade")
  from being answered using a barely-related retrieved entry.

Both of these are visible afterwards through `LAST_RETRIEVAL_INFO` (used by `chat()` for the `[memory]`
line) and are recorded per test case in the evaluation section.


In [18]:
# ================================================================
# History-aware retrieval + RAG chain
# ================================================================
LAST_RETRIEVAL_INFO = {
    "original_question": None,
    "rewritten_question": None,
    "used_rewrite": False,       # True whenever history existed and was used to shape the query
    "llm_rewrite_ok": None,      # True/False/None(no history) - did the LLM's own rewrite pass the guard?
    "best_score": None,          # best cosine similarity found for the (rewritten) question
    "below_threshold": False,    # True if the relevance gate refused before calling the LLM
}

BAD_REWRITE_MARKERS = ("ai:", "assistant:", "user:", "human:", "system:")


def sanitize_rewritten_query(raw_text, original_question):
    text = str(raw_text).strip()
    for marker in ["Standalone question:", "Rewritten question:", "Question:", "Answer:"]:
        if text.lower().startswith(marker.lower()):
            text = text[len(marker):].strip()
    text = text.split("\n")[0].strip().strip('"').strip()

    lowered = text.lower()
    looks_invented = any(marker in lowered for marker in BAD_REWRITE_MARKERS)
    too_short = len(text) < 6
    too_long = len(text) > 300 or len(text.split()) > 45
    not_question_shaped = not too_short and "?" not in text

    if too_short or too_long or looks_invented or not_question_shaped:
        return original_question, False
    return text, True


def build_fallback_question(current_question, history):
    # Used only when the LLM's own rewrite is rejected by the guard above. It is a
    # deliberately simple heuristic (no LLM call) that still folds the previous turn's
    # topic into the query, so retrieval has something better than the bare pronoun
    # question to match against.
    last_user_message = next(
        (message.content for message in reversed(history) if getattr(message, "type", "") == "human"),
        None,
    )
    if last_user_message:
        return f"{current_question} (previous question was: {last_user_message})"
    return current_question


def contextualize_question(inputs):
    original_question = inputs["input"]
    history = inputs.get("chat_history") or []

    LAST_RETRIEVAL_INFO["original_question"] = original_question

    if not history:
        LAST_RETRIEVAL_INFO.update(
            {"rewritten_question": original_question, "used_rewrite": False, "llm_rewrite_ok": None}
        )
        return original_question

    try:
        raw = (contextualize_prompt | llm | StrOutputParser()).invoke(inputs)
        raw_text = raw.content if hasattr(raw, "content") else str(raw)
        rewritten, llm_ok = sanitize_rewritten_query(raw_text, original_question)
    except Exception:
        rewritten, llm_ok = original_question, False

    if not llm_ok:
        rewritten = build_fallback_question(original_question, history)

    # A follow-up turn always gets *some* form of history-aware contextualisation
    # (either the LLM's own rewrite, or the heuristic fallback) - "used_rewrite" tracks
    # that memory was applied, "llm_rewrite_ok" tracks whether the LLM's rewrite itself
    # was good enough to use as-is.
    LAST_RETRIEVAL_INFO.update({"rewritten_question": rewritten, "used_rewrite": True, "llm_rewrite_ok": llm_ok})
    return rewritten


def history_aware_retrieve(inputs):
    rewritten_question = contextualize_question(inputs)

    scored = vectorstore.similarity_search_with_score(rewritten_question, k=max(RETRIEVER_K, 1))
    best_score = l2_to_cosine(min(distance for _, distance in scored)) if scored else 0.0
    LAST_RETRIEVAL_INFO["best_score"] = best_score

    if not scored or best_score < MIN_SIMILARITY:
        LAST_RETRIEVAL_INFO["below_threshold"] = True
        return []

    LAST_RETRIEVAL_INFO["below_threshold"] = False
    return retriever.invoke(rewritten_question)


history_aware_retriever = RunnableLambda(history_aware_retrieve)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

print("[OK] RAG chain built")
print("  history_aware_retriever : rewrite (if history) -> relevance gate -> FAISS retrieval")
print("  question_answer_chain   : retrieved entries -> grounding prompt -> LLM")
print("  rag_chain               : returns input, context (sources) and answer")


[OK] RAG chain built
  history_aware_retriever : rewrite (if history) -> relevance gate -> FAISS retrieval
  question_answer_chain   : retrieved entries -> grounding prompt -> LLM
  rag_chain               : returns input, context (sources) and answer


## 16. Adding Conversational Memory (buffer memory)

`RunnableWithMessageHistory` wraps the stateless `rag_chain` so that, on every call, it:

1. looks up the message history for the given `session_id`;
2. injects it into the `chat_history` placeholder of both prompts;
3. appends the new question and answer to that history after the call.

`InMemoryChatMessageHistory` keeps **every** turn of the session — no summarisation, no window, no
truncation. Different `session_id` values get independent buffers, which lets the test suite isolate
each scenario.


In [19]:
# ================================================================
# Conversational buffer memory
# ================================================================
SESSION_STORE = {}


def get_session_history(session_id):
    if session_id not in SESSION_STORE:
        SESSION_STORE[session_id] = InMemoryChatMessageHistory()
    return SESSION_STORE[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

print("[OK] Conversational RAG assistant ready")
print(f"Session id    : {SESSION_ID}")


[OK] Conversational RAG assistant ready
Session id    : midterm-demo


## 17. Interactive Chat Interface

* `chat(question)` — ask the assistant; prints the answer and the intents/categories used, and (for
  follow-ups) the rewritten query;
* `answer_question(question)` — a single-shot call with **no** memory, useful for comparing against `chat()`;
* `show_history()` — print the current buffer;
* `reset_memory()` — clear the buffer and start a fresh conversation.


In [20]:
# ================================================================
# Chat interface
# ================================================================
TURN_MARKER_RE = re.compile(r"\n\s*(Human|User|AI|Assistant|System)\s*:", re.IGNORECASE)


def format_sources(source_documents):
    seen = {}
    for document in source_documents:
        key = document.metadata.get("intent", "?")
        if key not in seen:
            seen[key] = document.metadata.get("category", "?")
    return [f"{intent} ({category})" for intent, category in seen.items()]


def clean_answer(raw_answer):
    text = raw_answer.content if hasattr(raw_answer, "content") else str(raw_answer)
    text = text.strip()
    # Belt-and-suspenders: even with ChatHuggingFace applying the real chat template,
    # cut the text at the first sign the model started inventing a new conversation turn.
    match = TURN_MARKER_RE.search(text)
    if match:
        text = text[: match.start()].strip()
    text = re.sub(r"^(AI|Assistant|Answer|System)\s*:\s*", "", text).strip()
    return text.strip()


def chat(question, session_id=SESSION_ID, show_sources=True, verbose=True):
    try:
        result = conversational_rag_chain.invoke(
            {"input": question},
            config={"configurable": {"session_id": session_id}},
        )
    except Exception as error:
        print("[ERROR] The assistant failed to answer:", error)
        return None

    context_docs = result.get("context", [])

    # Hard override: if the relevance gate found nothing good enough, the answer is
    # always the fixed refusal sentence, regardless of what the LLM produced. This makes
    # hallucination-avoidance a structural guarantee rather than something that depends
    # on a (possibly weak) local model choosing to follow the prompt's rule 2.
    if not context_docs or LAST_RETRIEVAL_INFO.get("below_threshold"):
        answer = REFUSAL_SENTENCE
    else:
        answer = clean_answer(result.get("answer", ""))
        if not answer:
            answer = REFUSAL_SENTENCE

    sources = format_sources(context_docs)

    if verbose:
        print("=" * 100)
        print(f"USER      : {question}")
        if LAST_RETRIEVAL_INFO["used_rewrite"]:
            note = "" if LAST_RETRIEVAL_INFO["llm_rewrite_ok"] else " (LLM rewrite rejected -> fallback used)"
            print(f"[memory]  : follow-up detected -> retrieved using "
                  f"\"{LAST_RETRIEVAL_INFO['rewritten_question']}\"{note}")
        print(f"[gate]    : best cosine similarity {LAST_RETRIEVAL_INFO['best_score']:.3f} "
              f"(threshold {MIN_SIMILARITY})")
        print("-" * 100)
        print("ASSISTANT :")
        print(textwrap.fill(answer, width=100, initial_indent="  ", subsequent_indent="  "))
        if show_sources and sources:
            print("\nGROUNDED IN (intent / category):")
            for source in sources:
                print(f"  - {source}")
        print("=" * 100 + "\n")

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "source_documents": context_docs,
        "used_memory": LAST_RETRIEVAL_INFO["used_rewrite"],
        "best_score": LAST_RETRIEVAL_INFO["best_score"],
        "refused": answer.strip() == REFUSAL_SENTENCE,
    }


def answer_question(question, verbose=True):
    # Single-shot RAG with NO memory - used to show what memory adds.
    result = rag_chain.invoke({"input": question, "chat_history": []})
    context_docs = result.get("context", [])
    if not context_docs or LAST_RETRIEVAL_INFO.get("below_threshold"):
        answer = REFUSAL_SENTENCE
    else:
        answer = clean_answer(result.get("answer", ""))
        if not answer:
            answer = REFUSAL_SENTENCE
    if verbose:
        print(f"[no memory] Q: {question}")
        print(textwrap.fill(answer, width=100, initial_indent="  ", subsequent_indent="  "))
    return answer


def show_history(session_id=SESSION_ID):
    history = get_session_history(session_id)
    if not history.messages:
        print(f"(session '{session_id}' has no messages yet)")
        return
    print(f"MESSAGE BUFFER — session '{session_id}' ({len(history.messages)} messages)")
    print("-" * 100)
    for message in history.messages:
        role = "USER" if message.type == "human" else "ASSISTANT"
        print(f"[{role}] {message.content}")


def reset_memory(session_id=SESSION_ID):
    SESSION_STORE[session_id] = InMemoryChatMessageHistory()
    print(f"[OK] Memory cleared for session '{session_id}'")


### 17.1 First conversation — RAG and memory working together

Turn 1 establishes a topic; turn 2 uses "that"; turn 3 uses "instead". Both follow-ups can only work if
the conversation history is used to rewrite the question before retrieval.


In [21]:
reset_memory()

_ = chat("How can I cancel my order?")
_ = chat("How long will that take?")
_ = chat("Can I get a refund instead?")


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[OK] Memory cleared for session 'midterm-demo'


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER      : How can I cancel my order?
[gate]    : best cosine similarity 0.977 (threshold 0.35)
----------------------------------------------------------------------------------------------------
ASSISTANT :
  To cancel your order, please follow these steps:  1. Sign into Your Account: Log in to your
  {{Online Company Portal Info}}. 2. Access Your Orders: Go to the '{{Online Order Interaction}}' or
  '{{Online Order Interaction}}' section. 3. Locate the Order: Look for the order with the number
  {{Order Number}} and click on it. 4. Initiate Cancelation: You will find a '{{Online Order
  Interaction}}' option associated with your purchase. Please select it.  If you need further
  assistance, feel free to ask!

GROUNDED IN (intent / category):
  - cancel_order (ORDER)



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER      : How long will that take?
[memory]  : follow-up detected -> retrieved using "Could you please specify which step you're referring to so I can provide more accurate information about the time frame?"
[gate]    : best cosine similarity 0.520 (threshold 0.35)
----------------------------------------------------------------------------------------------------
ASSISTANT :
  The cancellation process should typically take no longer than 2 business days once initiated.
  However, please note that the exact timeframe might vary depending on the status of your account
  and the current system load. If you experience delays, we recommend checking back periodically
  until the issue is resolved.

GROUNDED IN (intent / category):
  - set_up_shipping_address (SHIPPING)
  - switch_account (ACCOUNT)
  - check_invoice (INVOICE)
  - edit_account (ACCOUNT)



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER      : Can I get a refund instead?
[memory]  : follow-up detected -> retrieved using "Can I get a refund instead? (previous question was: How long will that take?)" (LLM rewrite rejected -> fallback used)
[gate]    : best cosine similarity 0.873 (threshold 0.35)
----------------------------------------------------------------------------------------------------
ASSISTANT :
  Yes, you can request a refund for your order. To proceed, simply contact the vendor through our
  website, email, or phone, and they will be happy to guide you through the process. Ensure you
  provide all necessary details such as your order number and reason for requesting a refund. We aim
  to resolve your concerns promptly, and your satisfaction is our top priority.

GROUNDED IN (intent / category):
  - get_refund (REFUND)
  - check_refund_policy (REFUND)



In [22]:
# The buffer now contains every turn of the conversation.
show_history()


MESSAGE BUFFER — session 'midterm-demo' (6 messages)
----------------------------------------------------------------------------------------------------
[USER] How can I cancel my order?
[ASSISTANT] To cancel your order, please follow these steps:

1. Sign into Your Account: Log in to your {{Online Company Portal Info}}.
2. Access Your Orders: Go to the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.
3. Locate the Order: Look for the order with the number {{Order Number}} and click on it.
4. Initiate Cancelation: You will find a '{{Online Order Interaction}}' option associated with your purchase. Please select it.

If you need further assistance, feel free to ask!
[USER] How long will that take?
[ASSISTANT] The cancellation process should typically take no longer than 2 business days once initiated. However, please note that the exact timeframe might vary depending on the status of your account and the current system load. If you experience delays, we recomme

## 18. Testing

Twelve test cases are defined below. Every question is written against **intents that actually exist**
in this dataset — except T8-T11, which are deliberately out of scope in order to test grounding, and T12,
which repeats one question to test reproducibility.

| # | Category | Question (final turn) | What it checks |
| --- | --- | --- | --- |
| 1 | Order cancellation | How can I cancel my order? | basic retrieval + factual answer |
| 2 | Refund | How do I get a refund for a damaged item? | retrieval of one self-contained entry |
| 3 | Shipping / tracking | How can I track my order? | correct-intent retrieval |
| 4 | Account | I forgot my password, how do I reset it? | retrieval from the account section |
| 5 | Payment | I was charged twice, what should I do? | retrieval from payment-issue intents |
| 6 | Follow-up (memory) | (after cancelling) *How long will that take?* | conversation memory used for retrieval |
| 7 | Pronoun follow-up | (after asking about refunds) *Can I do that for a subscription too?* | pronoun/topic resolution |
| 8 | Out of scope — unrelated | What is the weather like in Cairo tomorrow? | must refuse, not hallucinate |
| 9 | Out of scope — invented policy | Can you give me a $50 discount code right now? | must refuse; no such policy exists in the data |
| 10 | Out of scope — borderline discount | Can you give me a 90% discount on my order? | must refuse with the exact fixed sentence, not an improvised decline |
| 11 | Out of scope — unrelated request | Write a C++ program for me. | must refuse; coding help is unrelated to any of the 27 intents |
| 12 | Determinism (repeated question) | *Who are you?* asked twice, back to back | identical question -> identical answer, both times refused |

**Why we need it:** each test runs in its **own session id**, so a test's memory cannot leak into the next
test. For multi-turn tests, earlier turns run silently and only the final turn is evaluated (except T12,
where both turns are compared against each other to check reproducibility).


In [23]:
# ================================================================
# Test suite
# ================================================================
TEST_CASES = [
    {
        "id": "T1", "category": "Order cancellation",
        "turns": ["How can I cancel my order?"],
        "expected_behavior": "Explains the cancellation process from the approved response.",
        "expected_keywords": ["cancel", "order"],
        "mode": "grounded",
    },
    {
        "id": "T2", "category": "Refund",
        "turns": ["How do I get a refund for a damaged item?"],
        "expected_behavior": "Explains the refund process from the approved response.",
        "expected_keywords": ["refund", "money"],
        "mode": "grounded",
    },
    {
        "id": "T3", "category": "Shipping / tracking",
        "turns": ["How can I track my order?"],
        "expected_behavior": "Explains how to track an order.",
        "expected_keywords": ["track", "order", "delivery", "shipping"],
        "mode": "grounded",
    },
    {
        "id": "T4", "category": "Account",
        "turns": ["I forgot my password, how do I reset it?"],
        "expected_behavior": "Explains the password reset / recovery process.",
        "expected_keywords": ["password", "account", "reset"],
        "mode": "grounded",
    },
    {
        "id": "T5", "category": "Payment",
        "turns": ["I was charged twice, what should I do?"],
        "expected_behavior": "Explains how a payment issue is handled.",
        "expected_keywords": ["payment", "charge", "invoice"],
        "mode": "grounded",
    },
    {
        "id": "T6", "category": "Follow-up question (memory)",
        "turns": ["How can I cancel my order?", "How long will that take?"],
        "expected_behavior": "Uses the previous turn to understand 'that' and answers about cancellation timing.",
        "expected_keywords": ["cancel", "order"],
        "mode": "grounded",
    },
    {
        "id": "T7", "category": "Pronoun follow-up (memory)",
        "turns": ["How do I request a refund?", "Can I do that for a subscription too?"],
        "expected_behavior": "Resolves 'that' to 'request a refund' and answers about subscriptions/refunds.",
        "expected_keywords": ["refund", "subscription", "cancel"],
        "mode": "grounded",
    },
    {
        "id": "T8", "category": "Out of scope — unrelated",
        "turns": ["What is the weather like in Cairo tomorrow?"],
        "expected_behavior": "Refuses; weather is not a customer-support topic in this dataset.",
        "expected_keywords": [],
        "mode": "refusal",
    },
    {
        "id": "T9", "category": "Out of scope — invented policy",
        "turns": ["Can you give me a $50 discount code right now?"],
        "expected_behavior": "Refuses; no such promotion exists in the approved responses.",
        "expected_keywords": [],
        "mode": "refusal",
    },
    {
        "id": "T10", "category": "Out of scope — borderline discount request",
        "turns": ["Can you give me a 90% discount on my order?"],
        "expected_behavior": ("Refuses with the exact fixed sentence. This phrasing sounds like a "
                               "support request but weakly matches unrelated intents by wording alone "
                               "- this is the exact case that slipped past the old, looser relevance "
                               "gate and got an improvised (non-templated) decline instead."),
        "expected_keywords": [],
        "mode": "refusal",
    },
    {
        "id": "T11", "category": "Out of scope — unrelated request",
        "turns": ["Write a C++ program for me."],
        "expected_behavior": ("Refuses with the exact fixed sentence; writing code is unrelated to "
                               "any of the 27 customer-support intents in the knowledge base."),
        "expected_keywords": [],
        "mode": "refusal",
    },
    {
        "id": "T12", "category": "Determinism — repeated identical question",
        "turns": ["Who are you?", "Who are you?"],
        "expected_behavior": ("Asks the exact same out-of-scope question twice in one session. Both "
                               "answers must be identical, and both must correctly refuse - this is "
                               "the exact case where sampling (do_sample=True) previously produced two "
                               "different answers (one refusal, one hallucinated self-description) for "
                               "the same question."),
        "expected_keywords": [],
        "mode": "determinism",
    },
]


REFUSAL_MARKERS = ["could not find", "not available", "cannot help with that", "no approved answer"]


def run_test_suite(test_cases):
    results = []
    for case in test_cases:
        session_id = f"test-{case['id']}"
        reset_memory(session_id)

        turn_records = []
        for turn in case["turns"]:
            turn_records.append(chat(turn, session_id=session_id, verbose=False))
        record = turn_records[-1] if turn_records else None

        answer_lower = record["answer"].lower() if record else ""
        keywords_found = [kw for kw in case["expected_keywords"] if kw.lower() in answer_lower]
        refused = record["refused"] if record else any(marker in answer_lower for marker in REFUSAL_MARKERS)
        # A determinism case repeats the SAME question on purpose to check reproducibility -
        # it is not a follow-up/pronoun-resolution case, so it must not be counted as one.
        is_memory_case = len(case["turns"]) > 1 and case["mode"] != "determinism"
        used_memory = record["used_memory"] if record else False

        consistent = None
        if case["mode"] == "determinism":
            first_answer = turn_records[0]["answer"].strip() if turn_records and turn_records[0] else None
            second_answer = turn_records[-1]["answer"].strip() if turn_records and turn_records[-1] else None
            consistent = bool(first_answer) and first_answer == second_answer
            # Determinism must hold AND the (identical) answer must actually be correct
            # (a repeatable wrong answer is not a pass).
            passed = consistent and refused
        elif case["mode"] == "grounded":
            keyword_ok = bool(keywords_found) and bool(record["sources"]) and not refused
            # A follow-up test only really PASSes if the answer is right AND the memory
            # rewrite/fallback was actually applied. A keyword match that happens to be
            # right *without* using history is a coincidence, not evidence memory works -
            # this is exactly the contradiction the earlier version of this notebook had
            # (T6/T7 marked PASS while "used memory" was 0).
            passed = keyword_ok and (used_memory if is_memory_case else True)
        else:
            passed = refused

        results.append({
            "id": case["id"],
            "category": case["category"],
            "turns": case["turns"],
            "expected_behavior": case["expected_behavior"],
            "answer": record["answer"] if record else "",
            "sources": record["sources"] if record else [],
            "used_memory": used_memory,
            "is_memory_case": is_memory_case,
            "best_score": record["best_score"] if record else None,
            "keywords_found": keywords_found,
            "refused": refused,
            "mode": case["mode"],
            "consistent": consistent,
            "result": "PASS" if passed else "REVIEW",
        })
    return results


test_results = run_test_suite(TEST_CASES)

for record in test_results:
    print(f"[{record['result']}] {record['id']} — {record['category']}")


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[OK] Memory cleared for session 'test-T1'


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[OK] Memory cleared for session 'test-T2'


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[OK] Memory cleared for session 'test-T11'


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[OK] Memory cleared for session 'test-T12'


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[PASS] T1 — Order cancellation
[PASS] T2 — Refund
[PASS] T3 — Shipping / tracking
[PASS] T4 — Account
[PASS] T5 — Payment
[PASS] T6 — Follow-up question (memory)
[PASS] T7 — Pronoun follow-up (memory)
[PASS] T8 — Out of scope — unrelated
[PASS] T9 — Out of scope — invented policy
[REVIEW] T10 — Out of scope — borderline discount request
[PASS] T11 — Out of scope — unrelated request
[REVIEW] T12 — Determinism — repeated identical question


## 19. Evaluation

### How the `Result` column is computed — exactly

This is a **qualitative evaluation with a transparent automatic check**, not an accuracy percentage.

**For a grounded, single-turn test (T1–T5)** — `PASS` when all hold:

1. at least one **expected keyword** appears in the answer (case-insensitive substring match);
2. at least one **intent/category source** was retrieved and cited;
3. the answer is **not** a refusal.

**For a grounded, follow-up test (T6–T7)** — the same three conditions above, **plus** a fourth:

4. `used_memory` is `True`, i.e. the follow-up was actually rewritten (or the fallback heuristic was
   applied) using the conversation history before retrieval ran.

A correct-looking answer to a follow-up question that happened to work *without* using history is not
counted as evidence that memory works, so it does not get a `PASS` here — this is the fix for the earlier
contradiction where T6/T7 were marked `PASS` while their `used_memory` count was `0`.

**For an out-of-scope test (T8–T9)** — `PASS` when the assistant actually refused (checked against the
`refused` flag `chat()` returns, which is tied to the relevance gate, not just a keyword search over the
answer text).

Anything else is marked `REVIEW` — a human should read that row; it is not automatically a failure.


In [24]:
# ================================================================
# Evaluation table
# ================================================================
def build_evaluation_table(test_results, answer_chars=200):
    rows = []
    for record in test_results:
        rows.append({
            "Test": record["id"],
            "Category": record["category"],
            "Question": record["turns"][-1],
            "Expected Behavior": record["expected_behavior"],
            "Actual Answer": textwrap.shorten(record["answer"].replace("\n", " "), width=answer_chars,
                                              placeholder=" ..."),
            "Keywords Found": ", ".join(record["keywords_found"]) if record["keywords_found"] else "-",
            "Sources": ", ".join(record["sources"]) or "-",
            "Used Memory": "Yes" if record.get("used_memory") else "-",
            "Consistent": ("Yes" if record.get("consistent") else "No")
                          if record.get("consistent") is not None else "-",
            "Result": record["result"],
        })
    return pd.DataFrame(rows)


evaluation_table = build_evaluation_table(test_results)

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 250)

print("=" * 110)
print("EVALUATION TABLE  (qualitative - see the rule described above)")
print("=" * 110)
display(evaluation_table)

csv_path = WORKING_DIR / "evaluation_table.csv"
evaluation_table.to_csv(csv_path, index=False)
print(f"\nSaved to: {csv_path}")


EVALUATION TABLE  (qualitative - see the rule described above)


,Test,Category,Question,Expected Behavior,Actual Answer,Keywords Found,Sources,Used Memory,Consistent,Result
0,T1,Order cancellation,How can I cancel my order?,Explains the cancellation process from the approved response.,"To cancel your order, please follow these steps: 1. Sign into Your Account: Log in to your {{Online Company Portal Info}}. 2. Access Your Orders: Go to the '{{Online Order Interaction}}' or ...","cancel, order",cancel_order (ORDER),-,-,PASS
1,T2,Refund,How do I get a refund for a damaged item?,Explains the refund process from the approved response.,"To get a refund for a damaged item, follow these steps: 1. **Gather Documentation**: Collect all relevant documents such as proof of payment, receipt, invoice, and any communication regarding the ...",refund,"check_refund_policy (REFUND), get_refund (REFUND)",-,-,PASS
2,T3,Shipping / tracking,How can I track my order?,Explains how to track an order.,"To track your order, follow these steps: 1. Log in to your account on our website. 2. Navigate to the ""My Orders"" or ""Order History"" section. 3. Locate the specific order for which you are ...","track, order, delivery","track_order (ORDER), delivery_period (DELIVERY), track_refund (REFUND)",-,-,PASS
3,T4,Account,"I forgot my password, how do I reset it?",Explains the password reset / recovery process.,"Follow these steps: 1. Go to the login page of our platform. 2. Click on the ""Forgot Password"" option. 3. Enter the email address associated with your account. 4. Check your email for instructions ...","password, account, reset",recover_password (ACCOUNT),-,-,PASS
4,T5,Payment,"I was charged twice, what should I do?",Explains how a payment issue is handled.,"If you were charged twice for the same item or service, please contact our customer support team right away so we can resolve the issue and provide a refund.",charge,check_refund_policy (REFUND),-,-,PASS
5,T6,Follow-up question (memory),How long will that take?,Uses the previous turn to understand 'that' and answers about cancellation timing.,"The cancellation process should typically take no longer than 2 business days once initiated. However, please note that the exact timeframe might vary depending on the status of your account and ...",cancel,"set_up_shipping_address (SHIPPING), switch_account (ACCOUNT), check_invoice (INVOICE), edit_account (ACCOUNT)",Yes,-,PASS
6,T7,Pronoun follow-up (memory),Can I do that for a subscription too?,Resolves 'that' to 'request a refund' and answers about subscriptions/refunds.,"Yes, you can request a refund for a subscription. To do so, follow these steps: 1. Contact the vendor directly or use our customer support service. 2. Provide them with your order number and the ...","refund, subscription",newsletter_subscription (SUBSCRIPTION),Yes,-,PASS
7,T8,Out of scope — unrelated,What is the weather like in Cairo tomorrow?,Refuses; weather is not a customer-support topic in this dataset.,I could not find an approved answer for that in the support knowledge base.,-,-,-,-,PASS
8,T9,Out of scope — invented policy,Can you give me a $50 discount code right now?,Refuses; no such promotion exists in the approved responses.,I could not find an approved answer for that in the support knowledge base.,-,-,-,-,PASS
9,T10,Out of scope — borderline discount request,Can you give me a 90% discount on my order?,"Refuses with the exact fixed sentence. This phrasing sounds like a support request but weakly matches unrelated intents by wording alone - this is the exact case that slipped past the old, looser relevance gate and g...","I'm sorry, but I cannot fulfill your request for a 90% discount on your order. As per our terms and conditions, we do not offer discounts above 10%. However, if you're interested in purchasing ...",-,"place_order (ORDER), check_refund_policy (REFUND), get_refund (REFUND), track_order (ORDER)",-,-,REVIEW



Saved to: /content/evaluation_table.csv


In [25]:
# ================================================================
# Capability summary  (counts, not accuracy percentages)
# ================================================================
grounded_tests = [r for r in test_results if r["mode"] == "grounded"]
refusal_tests = [r for r in test_results if r["mode"] == "refusal"]
memory_tests = [r for r in test_results if r["is_memory_case"]]
determinism_tests = [r for r in test_results if r["mode"] == "determinism"]

summary_rows = [
    {
        "Capability": "Retrieved relevant information",
        "Measured by": "at least one intent cited for the answer",
        "Cases": len(test_results),
        "Met": sum(1 for r in test_results if r["sources"]),
    },
    {
        "Capability": "Answered from the knowledge base",
        "Measured by": "expected keyword present in the answer",
        "Cases": len(grounded_tests),
        "Met": sum(1 for r in grounded_tests if r["keywords_found"]),
    },
    {
        "Capability": "Used conversation history",
        "Measured by": "follow-up question rewritten (or fallback-merged) before retrieval",
        "Cases": len(memory_tests),
        "Met": sum(1 for r in memory_tests if r["used_memory"]),
    },
    {
        "Capability": "Follow-up answered correctly using memory",
        "Measured by": "memory case is a full PASS (right answer AND memory actually used)",
        "Cases": len(memory_tests),
        "Met": sum(1 for r in memory_tests if r["result"] == "PASS"),
    },
    {
        "Capability": "Avoided hallucination (out of scope)",
        "Measured by": "refusal marker present",
        "Cases": len(refusal_tests),
        "Met": sum(1 for r in refusal_tests if r.get("refused")),
    },
    {
        "Capability": "Reproducible on repeated identical questions",
        "Measured by": "the exact same question, asked twice in one session, gets the exact same answer",
        "Cases": len(determinism_tests),
        "Met": sum(1 for r in determinism_tests if r.get("consistent")),
    },
]

summary_table = pd.DataFrame(summary_rows)
print("=" * 110)
print("CAPABILITY SUMMARY - counts of cases meeting each explicitly defined check")
print(f"No accuracy percentage is claimed; these are counts over {len(test_results)} hand-written test cases.")
print("=" * 110)
display(summary_table)


CAPABILITY SUMMARY - counts of cases meeting each explicitly defined check
No accuracy percentage is claimed; these are counts over 12 hand-written test cases.


,Capability,Measured by,Cases,Met
0,Retrieved relevant information,at least one intent cited for the answer,12,9
1,Answered from the knowledge base,expected keyword present in the answer,7,7
2,Used conversation history,follow-up question rewritten (or fallback-merged) before retrieval,2,2
3,Follow-up answered correctly using memory,memory case is a full PASS (right answer AND memory actually used),2,2
4,Avoided hallucination (out of scope),refusal marker present,4,3
5,Reproducible on repeated identical questions,"the exact same question, asked twice in one session, gets the exact same answer",1,0


### 19.1 Retrieval Metrics

**What this cell does:** reports simple retrieval statistics for each test question: how many entries were
retrieved and their best/mean cosine similarity.

**Why we need it:** it separates *retrieval* quality from *generation* quality. If an answer is weak, this
table shows whether the retriever failed to find the right entry, or whether it succeeded and the LLM wrote
a poor answer.


In [26]:
# ================================================================
# Retrieval metrics
# ================================================================
def retrieval_report(questions, k=RETRIEVER_K):
    rows = []
    for question in questions:
        scored = vectorstore.similarity_search_with_score(question, k=k)
        similarities = [l2_to_cosine(distance) for _, distance in scored]
        intents = sorted({document.metadata["intent"] for document, _ in scored})
        rows.append({
            "Question": textwrap.shorten(question, width=58),
            "Entries retrieved": len(scored),
            "Best similarity": round(max(similarities), 3),
            "Mean similarity": round(sum(similarities) / len(similarities), 3),
            "Intents": ", ".join(intents),
        })
    return pd.DataFrame(rows)


retrieval_questions = [case["turns"][-1] if len(case["turns"]) == 1
                       else f"{case['turns'][0]} {case['turns'][-1]}"
                       for case in TEST_CASES]

metrics_table = retrieval_report(retrieval_questions)
print("=" * 110)
print("RETRIEVAL METRICS  (cosine similarity derived from FAISS L2 distance)")
print("=" * 110)
display(metrics_table)

print("\nInterpretation guide:")
print("  > 0.55  strong topical match - the retriever found the right intent")
print("  0.35-0.55  partial match - related content, answer may be incomplete")
print("  < 0.35  weak match - typically an out-of-scope question, where refusal is correct")


RETRIEVAL METRICS  (cosine similarity derived from FAISS L2 distance)


,Question,Entries retrieved,Best similarity,Mean similarity,Intents
0,How can I cancel my order?,4,0.977,0.920,cancel_order
1,How do I get a refund for a damaged item?,4,0.682,0.668,"check_refund_policy, get_refund"
2,How can I track my order?,4,0.824,0.807,"delivery_period, track_order, track_refund"
3,"I forgot my password, how do I reset it?",4,0.927,0.895,recover_password
4,"I was charged twice, what should I do?",4,0.761,0.684,check_refund_policy
5,How can I cancel my order? How long will that take?,4,0.955,0.877,cancel_order
6,How do I request a refund? Can I do that for a [...],4,0.821,0.803,"check_refund_policy, get_refund"
7,What is the weather like in Cairo tomorrow?,4,-0.281,-0.284,"check_refund_policy, delivery_options"
8,Can you give me a $50 discount code right now?,4,0.246,0.239,"change_order, place_order"
9,Can you give me a 90% discount on my order?,4,0.612,0.478,"check_refund_policy, place_order"



Interpretation guide:
  > 0.55  strong topical match - the retriever found the right intent
  0.35-0.55  partial match - related content, answer may be incomplete
  < 0.35  weak match - typically an out-of-scope question, where refusal is correct


## 20. Final Project Demonstration

Runs one continuous, realistic customer conversation from a clean memory buffer, demonstrating the three
project pillars in a single unbroken exchange: **RAG** (every answer is grounded in a retrieved intent),
**memory** (turns 2–4 use pronouns/references that only resolve through history), and **grounding** (the
final turn asks for something the dataset does not contain, and the assistant declines instead of
inventing it).


In [27]:
# ================================================================
# Final demonstration conversation
# ================================================================
DEMO_SESSION = "final-demo"
reset_memory(DEMO_SESSION)

demo_questions = [
    "I want to cancel an order I just placed",
    "How long will that take to process?",
    "If it's already shipped, can I still get a refund?",
    "Can you also give me a free upgrade for the trouble?",
]

for demo_question in demo_questions:
    _ = chat(demo_question, session_id=DEMO_SESSION)


[OK] Memory cleared for session 'final-demo'


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER      : I want to cancel an order I just placed
[gate]    : best cosine similarity 0.957 (threshold 0.35)
----------------------------------------------------------------------------------------------------
ASSISTANT :
  I've picked up that you recently made a purchase and now would like to cancel the order with the
  purchase number {{Order Number}}. I apologize for any inconvenience this may cause. To assist you
  with the cancellation process, please follow these steps:  1. Log in to your {{Online Company
  Portal Info}} using your username and password. 2. Navigate to your order history or account
  settings page. 3. Locate the specific order with the order number {{Order Number}}. 4. Click on
  the order to access the details. 5. Look for the option to cancel the order and select it. 6.
  Confirm the cancellation and follow any additional prompts if necessary.

GROUNDED IN (intent / category):
  - cancel_order (ORDER)



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER      : How long will that take to process?
[memory]  : follow-up detected -> retrieved using "How long will the cancellation process take to complete?"
[gate]    : best cosine similarity 0.771 (threshold 0.35)
----------------------------------------------------------------------------------------------------
ASSISTANT :
  The processing time for your cancellation can vary depending on the status of your order and the
  current workload of our systems. Typically, we aim to process cancellations within 2 business
  days. However, please note that there might be some delays due to system maintenance or other
  unforeseen circumstances. We appreciate your patience while we handle your request. Once your
  cancellation has been processed, you will receive a confirmation email. Please check your inbox
  for more information.

GROUNDED IN (intent / category):
  - cancel_order (ORDER)
  - check_refund_policy (REFUND)



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER      : If it's already shipped, can I still get a refund?
[memory]  : follow-up detected -> retrieved using "If it's already shipped, can I still get a refund? (previous question was: How long will that take to process?)" (LLM rewrite rejected -> fallback used)
[gate]    : best cosine similarity 0.735 (threshold 0.35)
----------------------------------------------------------------------------------------------------
ASSISTANT :
  Unfortunately, once an order has been shipped, it cannot be canceled or refunded. The shipping
  process involves multiple parties involved in delivering the goods to you, including logistics
  companies and carriers. Therefore, once an order is shipped, it is considered final and cannot be
  reversed without causing significant disruption to those involved in the delivery process.
  However, if you experience issues with the shipment, such as non-delivery or damage during
  transit, you should reach out to the carrier or the seller directly for assistan

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER      : Can you also give me a free upgrade for the trouble?
[memory]  : follow-up detected -> retrieved using "Can you also give me a free upgrade for the trouble? (previous question was: If it's already shipped, can I still get a refund?)" (LLM rewrite rejected -> fallback used)
[gate]    : best cosine similarity 0.498 (threshold 0.35)
----------------------------------------------------------------------------------------------------
ASSISTANT :
  I'm sorry, but at this moment, we don't have the capability to provide a free upgrade for any
  issue related to your order. Our primary focus is to ensure that customers receive high-quality
  products and services. If you encounter any problems with your order, please contact the seller or
  the online retailer directly for assistance. They may be able to offer solutions or refunds based
  on their policies.

GROUNDED IN (intent / category):
  - check_refund_policy (REFUND)
  - switch_account (ACCOUNT)
  - track_refund (REFUND)



In [28]:
show_history(DEMO_SESSION)


MESSAGE BUFFER — session 'final-demo' (8 messages)
----------------------------------------------------------------------------------------------------
[USER] I want to cancel an order I just placed
[ASSISTANT] I've picked up that you recently made a purchase and now would like to cancel the order with the purchase number {{Order Number}}. I apologize for any inconvenience this may cause. To assist you with the cancellation process, please follow these steps:

1. Log in to your {{Online Company Portal Info}} using your username and password.
2. Navigate to your order history or account settings page.
3. Locate the specific order with the order number {{Order Number}}.
4. Click on the order to access the details.
5. Look for the option to cancel the order and select it.
6. Confirm the cancellation and follow any additional prompts if necessary.
[USER] How long will that take to process?
[ASSISTANT] The processing time for your cancellation can vary depending on the status of your order 

### 20.1 Why memory matters — a controlled comparison

Asks the same follow-up question twice: once with no memory (`answer_question`) and once inside the
demonstration conversation (`chat`). Without history, "How long will that take?" has no referent.


In [29]:
print("WITHOUT MEMORY (stateless RAG)")
print("-" * 100)
_ = answer_question("How long will that take?")

print("\n\nWITH MEMORY (conversational RAG, continuing the demo conversation)")
print("-" * 100)
_ = chat("How long will that take?", session_id=DEMO_SESSION)


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WITHOUT MEMORY (stateless RAG)
----------------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[no memory] Q: How long will that take?
  I could not find an approved answer for that in the support knowledge base.


WITH MEMORY (conversational RAG, continuing the demo conversation)
----------------------------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER      : How long will that take?
[memory]  : follow-up detected -> retrieved using "How long will that take? (previous question was: Can you also give me a free upgrade for the trouble?)" (LLM rewrite rejected -> fallback used)
[gate]    : best cosine similarity 0.593 (threshold 0.35)
----------------------------------------------------------------------------------------------------
ASSISTANT :
  Once you contact the seller or the online retailer regarding the issue, they will typically
  respond promptly to address your concerns. It generally takes them around 2-3 business days to
  review your case and either resolve the issue or provide a satisfactory solution. Please keep in
  mind that the actual timeframe may depend on various factors, such as the volume of orders being
  handled by the seller and the complexity of the situation. If you need further assistance or
  updates, feel free to reach out again.

GROUNDED IN (intent / category):
  - switch_account (ACCOUNT)



## 21. Interface — In-Notebook Gradio Chat

**What this cell does:** wraps the same `chat()` function in a small Gradio `ChatInterface`, so the
assistant can be used as a product rather than a sequence of function calls, with no tunnel or separate
server process required.

**How to use it:** run the cell, then type a question in the box that appears below it. Ask a follow-up
using "that" or "it" — the printed logs above already show the `[memory]` line; the widget itself shows the
clean conversation.


In [30]:
# ================================================================
# Gradio chat interface
# ================================================================
import gradio as gr

UI_SESSION = "gradio-chat"


def gradio_respond(message, history):
    result = chat(message, session_id=UI_SESSION, verbose=False)
    if result is None:
        return "Sorry, something went wrong while answering. Please try again."
    return result["answer"]


reset_memory(UI_SESSION)

demo_ui = gr.ChatInterface(
    fn=gradio_respond,
    title="🎧 Customer Support Knowledge Assistant",
    description=(
        "Ask a customer-support question (orders, refunds, shipping, accounts, payments...). "
        "Answers are grounded in the Bitext customer-support knowledge base."
    ),
    examples=[
        "How do I cancel my order?",
        "How can I get a refund?",
        "I forgot my account password",
        "What's today's weather in Cairo?",
    ],
)

# demo_ui.launch(share=True)  # uncomment to launch with a public link
demo_ui.launch()


[OK] Memory cleared for session 'gradio-chat'
* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://830e48b8c2d186e09f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 22. Limitations

This system is a working mid-term prototype, not a finished product. Its honest limits:

**Knowledge limits**

* The assistant knows **only** what is in the sampled Bitext knowledge base. Anything outside customer
  support (or outside the 27 covered intents) is simply unavailable, and by design the assistant says so
  rather than guessing.
* The dataset's responses are **templated** (`{{Order Number}}`, `{{Refund Amount}}`, etc.) — there is
  **no live account data**, no real order lookup, no actual payment processing.
* Only `MAX_EXAMPLES_PER_INTENT` examples per intent are indexed, not the full 26,872 rows, to keep the
  index small and fast to build. Increasing this constant broadens phrasing coverage at the cost of a
  larger, slower-to-build index.

**Technical limits**

* **Retrieval is capped at `k = 4` entries.** A question that spans multiple intents at once may only get
  a partial answer.
* **The LLM can still make mistakes.** Grounding greatly reduces hallucination but does not eliminate it —
  a small model can still merge two entries or over-generalise. On CPU (`flan-t5-base`), answers are
  noticeably terser than on a GPU model.
* **Memory is in-memory only.** Restarting the notebook erases the conversation; there is no persistence
  across sessions.
* **Evaluation is qualitative.** Twelve hand-written test cases with a keyword check is a sanity test, not a
  benchmark. No accuracy figure is claimed.

**Realistic improvements**

Hybrid retrieval (BM25 + vectors), a re-ranking model, an intent classifier used to pre-filter the
retriever, connecting the assistant to a real order/account API instead of templated text, and a larger
evaluation set with human relevance labels.


## 23. Final Project Checklist

**What this cell does:** verifies each project requirement against the live state of the notebook, instead
of printing a hard-coded list of ticks.


In [31]:
# ================================================================
# Verified project checklist
# ================================================================
memory_test_results = [r for r in test_results if r["is_memory_case"]]
refusal_test_results = [r for r in test_results if r["mode"] == "refusal"]

checks = [
    ("Dataset loading", len(df) > 0, f"{len(df):,} rows loaded from {DATASET_NAME}"),
    ("Knowledge-base construction", len(documents) > 0,
     f"{len(documents):,} documents built across {df['intent'].nunique()} intents"),
    ("Chunking / safety-net splitting", len(chunks) > 0,
     f"{len(chunks):,} chunks, size {CHUNK_SIZE}, overlap {CHUNK_OVERLAP}"),
    ("HuggingFace embeddings", len(sample_vector) > 0,
     f"{EMBEDDING_MODEL_NAME}, {len(sample_vector)} dimensions"),
    ("FAISS vector database", vectorstore.index.ntotal == len(chunks),
     f"{vectorstore.index.ntotal} vectors indexed"),
    ("Semantic retrieval", len(retriever.invoke("refund")) > 0,
     f"search_type={RETRIEVER_SEARCH_TYPE}, k={RETRIEVER_K}"),
    ("LLM integration", llm is not None, LLM_DESCRIPTION),
    ("LLM called through its chat template (no invented turns)", LLM_IS_CHAT,
     "ChatHuggingFace applied" if LLM_IS_CHAT else "plain-text model (flan-t5) or fallback in use - no chat template needed/available"),
    ("RAG pipeline", rag_chain is not None, "retriever -> grounding prompt -> LLM"),
    ("Conversation memory", len(get_session_history(DEMO_SESSION).messages) > 0,
     f"buffer memory, {len(get_session_history(DEMO_SESSION).messages)} messages in the demo session"),
    ("Remembers previous questions / follow-ups used",
     bool(memory_test_results) and all(r["used_memory"] for r in memory_test_results),
     f"{sum(r['used_memory'] for r in memory_test_results)}/{len(memory_test_results)} follow-ups rewritten or fallback-merged with history"),
    ("Follow-up questions answered correctly",
     bool(memory_test_results) and all(r["result"] == "PASS" for r in memory_test_results),
     f"{sum(r['result'] == 'PASS' for r in memory_test_results)}/{len(memory_test_results)} memory tests fully PASS (correct answer AND memory used)"),
    ("Source / intent citations", all(r["sources"] or r["mode"] == "refusal" for r in test_results),
     "intent + category printed with every grounded answer"),
    ("Testing", len(test_results) == len(TEST_CASES), f"{len(test_results)} test cases executed"),
    ("Evaluation", len(evaluation_table) == len(TEST_CASES), "evaluation table + capability summary"),
    ("Hallucination handling",
     bool(refusal_test_results) and all(r.get("refused") for r in refusal_test_results),
     f"{sum(r.get('refused') for r in refusal_test_results)}/{len(refusal_test_results)} out-of-scope questions refused "
     f"(relevance gate at cosine < {MIN_SIMILARITY}, not just LLM discretion)"),
    ("Interface", True, "Gradio ChatInterface wired to the same conversational RAG chain"),
]

print("=" * 100)
print("FINAL PROJECT CHECKLIST")
print("=" * 100)
for label, passed, evidence in checks:
    mark = "[x]" if passed else "[ ]"
    print(f"{mark} {label:<52} | {evidence}")
print("=" * 100)
print(f"{sum(1 for _, passed, _ in checks if passed)} / {len(checks)} requirements verified in this run")


FINAL PROJECT CHECKLIST
[x] Dataset loading                                      | 26,872 rows loaded from bitext/Bitext-customer-support-llm-chatbot-training-dataset
[x] Knowledge-base construction                          | 1,080 documents built across 27 intents
[x] Chunking / safety-net splitting                      | 3,527 chunks, size 400, overlap 60
[x] HuggingFace embeddings                               | sentence-transformers/all-MiniLM-L6-v2, 384 dimensions
[x] FAISS vector database                                | 3527 vectors indexed
[x] Semantic retrieval                                   | search_type=mmr, k=4
[x] LLM integration                                      | Qwen/Qwen2.5-1.5B-Instruct (causal, cuda, via ChatHuggingFace chat template)
[x] LLM called through its chat template (no invented turns) | ChatHuggingFace applied
[x] RAG pipeline                                         | retriever -> grounding prompt -> LLM
[x] Conversation memory                        

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 24. Conclusion

This notebook implements a complete Retrieval-Augmented Generation assistant with conversational memory,
built on the **Bitext Customer Support LLM Chatbot Training Dataset** instead of a PDF knowledge base.

**What was built**

Each of the dataset's `(instruction, category, intent, response)` rows is treated as one atomic knowledge
unit — a de-duplicated, capped sample per intent is embedded with the free `all-MiniLM-L6-v2`
sentence-transformer and indexed in FAISS. A LangChain MMR retriever returns the most relevant and mutually
distinct support entries for a question. Those entries are inserted into a prompt that forbids outside
knowledge and invented policy details, and an LLM writes a short, professional support reply that names
the intent/category it used.

**What makes it conversational**

Memory is placed *before* retrieval, not after it. Every turn is stored in a full buffer
(`InMemoryChatMessageHistory` + `RunnableWithMessageHistory`), and before each retrieval the current
question is rewritten into a standalone question using that history. This is what lets "how long will that
take?" work, and it is the difference between a demo that looks good in one turn and one that holds up in a
real conversation.

**What was learned**

* Structured, row-based datasets need a different "chunking" strategy than PDFs: the atomic unit is already
  defined by the data (one row = one knowledge unit), and the real engineering problem shifts to *sampling
  and de-duplication* rather than text splitting.
* Refusing is a feature. A support bot that says "I could not find an approved answer for that" is more
  trustworthy than one that invents a discount code or a refund policy that was never approved.
* Follow-up handling is where naive RAG breaks, and fixing it costs one extra LLM call before retrieval.

**Where it would go next:** an intent classifier to pre-filter retrieval, hybrid BM25 + vector search, a
re-ranker, and wiring the assistant to a real order/account backend instead of templated responses.
